In [ ]:
zad 1

In [3]:
import numpy as np

rng = np.random.default_rng(seed=42)

data = rng.normal(
    loc=50,
    scale=10,
    size=1000
)
data

array([53.0471708 , 39.60015894, 57.50451196, 59.40564716, 30.48964811,
       36.97820493, 51.27840403, 46.83757408, 49.83198842, 41.46956072,
       58.79397975, 57.77791935, 50.66030698, 61.27241207, 54.67509342,
       41.40707537, 53.68750784, 40.41117399, 58.78450301, 49.50074089,
       48.15137636, 43.19070456, 62.22541339, 48.45470518, 45.71672178,
       46.4786645 , 55.32309186, 53.65444064, 54.12732612, 54.30821003,
       71.41647601, 45.93584984, 44.87757271, 41.86227272, 56.15979423,
       61.28972293, 48.86052542, 41.59843523, 41.75518784, 56.50592788,
       57.43254171, 55.43154268, 43.34490293, 52.32161323, 51.16685809,
       52.18688597, 58.71428778, 52.23595549, 56.78913563, 50.67579069,
       52.89119399, 56.31288226, 35.4284418 , 46.80328784, 45.29627346,
       43.61122152, 47.24857749, 64.94941311, 41.34168884, 59.68278355,
       33.17130228, 46.6511497 , 51.62753065, 55.86222331, 57.1122658 ,
       57.93347235, 46.51274928, 45.37648207, 58.57975881, 48.08

In [7]:
empirical_percent = ((data >= 40) & (data <= 60)).mean() * 100
empirical_percent - 68.27

np.float64(0.5300000000000011)

In [12]:
def ascii_histogram(data, bins=30, height=10, label="Histogram"):

    counts, edges = np.histogram(data, bins=bins)

    max_count = counts.max() if counts.max() > 0 else 1

    scaled = np.round(counts / max_count * height).astype(int)

    print(f"\n{label} (max={max_count}):")

    for h in range(height, 0, -1):

        row = "".join("█" if s >= h else " " for s in scaled)

        print(f" {row}")

    print(" " + "─" * bins)

    print(f" {edges[0]:>5.1f}{' ' * (bins - 12)}{edges[-1]:>5.1f}")

In [14]:
ascii_histogram(
    data,
    bins=20,
    height=10,
    label="N(50,10)"
)


N(50,10) (max=160):
            █        
            █        
          ███        
          ███        
         █████       
        ███████      
        ███████      
       █████████     
      ███████████    
    ██████████████   
 ────────────────────
  13.5         81.8


In [ ]:
zad 2

In [16]:
from math import erf

def _erf_vec(x):
    return np.vectorize(erf)(x)

def normal_cdf(x, mu, sigma):
    """Dystrybuanta rozkładu normalnego przez math.erf."""
    return 0.5 * (1.0 + _erf_vec((x - mu) / (sigma * np.sqrt(2))))

P_X_le_85 = normal_cdf(85, 100, 15)

P_X_ge_120 = 1 - normal_cdf(120, 100, 15)

P_90_le_X_le_110 = (
    normal_cdf(110, 100, 15)
    - normal_cdf(90, 100, 15)
)

print(f"P(X ≤ 85) = {P_X_le_85:.4f}")
print(f"P(X ≥ 120) = {P_X_ge_120:.4f}")
print(f"P(90 ≤ X ≤ 110) = {P_90_le_X_le_110:.4f}")


P(X ≤ 85) = 0.1587
P(X ≥ 120) = 0.0912
P(90 ≤ X ≤ 110) = 0.4950


In [ ]:
zad 3

In [17]:
def make_synthetic_flowers(seed=42):
    """Generuje syntetyczny dataset kwiatów (150 x 4) z 3 gatunkami."""
    rng = np.random.default_rng(seed)
    n_per_class = 50
    centers = np.array([
        [5.0, 3.4, 1.5, 0.2],
        [5.9, 2.8, 4.3, 1.3],
        [6.6, 3.0, 5.6, 2.0],
    ])
    stds = np.array([0.35, 0.35, 0.45, 0.25])
    X_list, y_list = [], []
    for class_idx, center in enumerate(centers):
        samples = rng.normal(loc=center, scale=stds, size=(n_per_class, 4))
        X_list.append(samples)
        y_list.append(np.full(n_per_class, class_idx))
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
    target_names = np.array(['gatunek_A', 'gatunek_B', 'gatunek_C'])
    return X, y, feature_names, target_names

In [22]:
X, y, feature_names, target_names = make_synthetic_flowers()

In [24]:
import numpy as np
import math

def skewness(x):
    mean = np.mean(x)
    std = np.std(x, ddof=0)

    return np.mean(((x - mean) / std) ** 3)


def kurtosis_excess(x):
    mean = np.mean(x)
    std = np.std(x, ddof=0)

    return np.mean(((x - mean) / std) ** 4) - 3

In [25]:
import math

def jarque_bera_manual(x):
    n = len(x)
    sk = skewness(x)
    ku = kurtosis_excess(x)

    jb = n / 6.0 * (sk ** 2 + (ku ** 2) / 4.0)

    # Chi-square(2): p-value = exp(-JB/2)
    p_value = math.exp(-jb / 2)

    return jb, p_value, sk, ku


print("\n" + "=" * 60)
print("TEST NORMALNOŚCI (Jarque-Bera - ręczna implementacja)")
print("=" * 60)

for i, name in enumerate(feature_names):
    col = X[:, i]

    jb, p_value, sk, ku = jarque_bera_manual(col)

    verdict = "JEST normalny" if p_value > 0.05 else "NIE JEST normalny"

    print(f"\n{name}:")
    print(f" JB statystyka = {jb:.4f}, p-value = {p_value:.4f}")
    print(f" Skewness: {sk:+.4f} (0=symetria)")
    print(f" Kurtosis (excess): {ku:+.4f} (0=jak normalny)")
    print(f" → Rozkład {verdict}")


TEST NORMALNOŚCI (Jarque-Bera - ręczna implementacja)

sepal_length:
 JB statystyka = 7.5351, p-value = 0.0231
 Skewness: -0.0666 (0=symetria)
 Kurtosis (excess): -1.0899 (0=jak normalny)
 → Rozkład NIE JEST normalny

sepal_width:
 JB statystyka = 1.0293, p-value = 0.5977
 Skewness: -0.1215 (0=symetria)
 Kurtosis (excess): -0.3250 (0=jak normalny)
 → Rozkład JEST normalny

petal_length:
 JB statystyka = 15.0040, p-value = 0.0006
 Skewness: -0.3997 (0=symetria)
 Kurtosis (excess): -1.3272 (0=jak normalny)
 → Rozkład NIE JEST normalny

petal_width:
 JB statystyka = 10.8238, p-value = 0.0045
 Skewness: -0.2138 (0=symetria)
 Kurtosis (excess): -1.2446 (0=jak normalny)
 → Rozkład NIE JEST normalny


In [ ]:
zad 4

In [63]:
def pearson_corr(x, y):
    """Ręczny Pearson: r = cov(x,y) / (std(x) * std(y))."""

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    xm, ym = x.mean(), y.mean()

    cov = ((x - xm) * (y - ym)).mean()

    return cov / (x.std(ddof=0) * y.std(ddof=0))


def corr_pvalue_approx(r, n):
    """Aproksymacja normalna p-value dla Pearson r lub Spearman ρ."""

    t = r * np.sqrt((n - 2) / max(1 - r ** 2, 1e-12))

    return 2 * (
        1 - 0.5 * (
            1 + float(_erf_vec(abs(t) / np.sqrt(2)))
        )
    )


def ascii_scatter(x, y, title, rows=14, cols=50):

    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()

    x_range = max(x_max - x_min, 1e-9)
    y_range = max(y_max - y_min, 1e-9)

    grid = [[" "] * cols for _ in range(rows)]

    # Punkty
    for xi, yi in zip(x, y):

        c = int(
            (xi - x_min) / x_range * (cols - 1)
        )

        r_ = rows - 1 - int(
            (yi - y_min) / y_range * (rows - 1)
        )

        if grid[r_][c] == " ":
            grid[r_][c] = "·"
        elif grid[r_][c] == "·":
            grid[r_][c] = "*"
        else:
            grid[r_][c] = "#"

    # Linia regresji
    slope, intercept = np.polyfit(x, y, 1)

    for c in range(cols):

        xi = x_min + c / (cols - 1) * x_range
        yi = slope * xi + intercept

        r_ = rows - 1 - int(
            (yi - y_min) / y_range * (rows - 1)
        )

        if 0 <= r_ < rows and grid[r_][c] in (" ", "·"):
            grid[r_][c] = "─"

    print(f"\n{title}")
    print(f"y = {slope:.3f}·x + {intercept:.3f}")

    for row in grid:
        print(" |" + "".join(row) + "|")

    print(" " + "─" * (cols + 2))
    print(
        f" x: [{x_min:.2f}, {x_max:.2f}] "
        f"y: [{y_min:.2f}, {y_max:.2f}]"
    )

In [ ]:

X, y, feature_names, target_names = make_synthetic_flowers()


x = X[:, 2]
y = X[:, 3]


r = pearson_corr(x, y)


n = len(x)
p_value = corr_pvalue_approx(r, n)


print(f"Pearson r = {r:.4f}")
print(f"p-value = {p_value:.4f}")


ascii_scatter(
    x,
    y,
    title="petal_length vs petal_width",
    rows=20,
    cols=60
)

Pearson r = 0.9376
p-value = 0.0000

petal_length vs petal_width
y = 0.418·x + -0.426
 |                                                ·           |
 |                                                            |
 |                                               ·    *      ─|
 |                                               ·* ·     ─── |
 |                                         · ··* *· ··────   ·|
 |                                         ··    ·**── ·*     |
 |                                      *    #  *── · *       |
 |                               ·  · ·  ·  ───* · *          |
 |                                  ·· · ──*       ·          |
 |                            ·   ··*─#──·*##                 |
 |                          ·  ·  ───  · *   ·                |
 |                             ───*  ···· ·                   |
 |                 ·       ────·  ·  ··   *                   |
 |                      ───                                   |
 |        ·       

In [ ]:
zad 5

In [ ]:
import pandas as pd
import numpy as np

# === GENEROWANIE SYNTETYCZNEGO DATASETU CALIFORNIA HOUSING ===
def make_california_housing(n=5000, seed=42):
    """Generuje syntetyczny dataset wzorowany na California Housing."""
    
    rng = np.random.default_rng(seed)

    # Współrzędne geograficzne
    latitude = rng.uniform(32.5, 41.5, n)
    longitude = rng.uniform(-123.5, -114.5, n)

    # Dochód: rozkład skośny
    med_inc = rng.gamma(shape=2.0, scale=1.8, size=n) + 0.5

    # Wiek domu
    house_age = rng.integers(1, 55, n)

    # Pokoje / sypialnie / populacja
    ave_rooms = rng.normal(5.5, 1.5, n).clip(1, 15)
    ave_bedrms = (ave_rooms * rng.uniform(0.15, 0.25, n)).clip(0.5, 5)
    population = rng.integers(50, 5000, n)
    ave_occup = rng.normal(3.0, 0.8, n).clip(1, 10)

    # Cena domu
    med_house_val = (
        med_inc * 0.6
        + (latitude - 36) * 0.15
        - (house_age / 100)
        + rng.normal(0, 0.3, n)
    ).clip(0.15, 5.0)

    return pd.DataFrame({
        'MedInc': med_inc.round(4),
        'HouseAge': house_age,
        'AveRooms': ave_rooms.round(4),
        'AveBedrms': ave_bedrms.round(4),
        'Population': population,
        'AveOccup': ave_occup.round(4),
        'Latitude': latitude.round(4),
        'Longitude': longitude.round(4),
        'MedHouseVal': med_house_val.round(4)
    })




=== Syntetyczny California Housing Dataset ===
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  0.9067        48    4.1561     0.9944        3929    2.5674   39.4656   
1  7.1731        51    6.9524     1.3357        3825    4.9413   36.4499   
2  4.8817        22    5.0764     0.9349        3550    3.2407   40.2274   
3  4.0980        54    3.3280     0.8064        3875    2.5277   38.7763   
4  5.1915         5    6.3899     1.1873        4516    3.7003   33.3476   

   Longitude  MedHouseVal  
0  -121.5367       0.4077  
1  -117.5337       4.0278  
2  -116.5982       3.2236  
3  -122.0003       3.0991  
4  -123.1053       2.2712  


In [73]:
df = make_california_housing(n=2000, seed=42)


In [ ]:
C = np.corrcoef(df.values.T)
C

array([[ 1.        , -0.00878967,  0.01926581,  0.01177061,  0.00825595,
        -0.03343156,  0.01767043,  0.02565535,  0.90755379],
       [-0.00878967,  1.        ,  0.02563232,  0.02943039,  0.03017518,
         0.01356082, -0.00131199, -0.01913276, -0.10995291],
       [ 0.01926581,  0.02563232,  1.        ,  0.87507484, -0.03459028,
         0.04792139, -0.01911767,  0.02360194,  0.00538515],
       [ 0.01177061,  0.02943039,  0.87507484,  1.        , -0.03082111,
         0.02926402, -0.01606206,  0.02232911, -0.00564844],
       [ 0.00825595,  0.03017518, -0.03459028, -0.03082111,  1.        ,
         0.01470136,  0.0174733 , -0.01594834,  0.01164387],
       [-0.03343156,  0.01356082,  0.04792139,  0.02926402,  0.01470136,
         1.        ,  0.02380365,  0.00499435, -0.02419164],
       [ 0.01767043, -0.00131199, -0.01911767, -0.01606206,  0.0174733 ,
         0.02380365,  1.        , -0.00470459,  0.26796026],
       [ 0.02565535, -0.01913276,  0.02360194,  0.02232911, -0

In [ ]:
def corr_to_char(r):
    if r >= 0.75:
        return "█"
    elif r >= 0.5:
        return "▓"
    elif r >= 0.25:
        return "▒"
    elif r >= 0.05:
        return "░"
    elif r > -0.05:
        return "·"
    elif r > -0.25:
        return "-"
    elif r > -0.5:
        return "/"
    else:
        return "#"


def ascii_heatmap(M, labels):
    n = len(labels)

    short = [l[:7] for l in labels]

    print("\nASCII heatmap macierzy korelacji:")
    
    header = " " * 9 + "".join(f"{s:>7}" for s in short)
    print(header)

    for i in range(n):
        row = f"{short[i]:>8} "

        for j in range(n):
            row += f" {corr_to_char(M[i, j])} {M[i, j]:+.2f}"[:7]

        print(row)





ASCII heatmap macierzy korelacji:
          MedIncHouseAgAveRoomAveBedrPopulatAveOccuLatitudLongituMedHous
  MedInc  █ +1.0 · -0.0 · +0.0 · +0.0 · +0.0 · -0.0 · +0.0 · +0.0 █ +0.9
 HouseAg  · -0.0 █ +1.0 · +0.0 · +0.0 · +0.0 · +0.0 · -0.0 · -0.0 - -0.1
 AveRoom  · +0.0 · +0.0 █ +1.0 █ +0.8 · -0.0 · +0.0 · -0.0 · +0.0 · +0.0
 AveBedr  · +0.0 · +0.0 █ +0.8 █ +1.0 · -0.0 · +0.0 · -0.0 · +0.0 · -0.0
 Populat  · +0.0 · +0.0 · -0.0 · -0.0 █ +1.0 · +0.0 · +0.0 · -0.0 · +0.0
 AveOccu  · -0.0 · +0.0 · +0.0 · +0.0 · +0.0 █ +1.0 · +0.0 · +0.0 · -0.0
 Latitud  · +0.0 · -0.0 · -0.0 · -0.0 · +0.0 · +0.0 █ +1.0 · -0.0 ▒ +0.2
 Longitu  · +0.0 · -0.0 · +0.0 · +0.0 · -0.0 · +0.0 · -0.0 █ +1.0 · +0.0
 MedHous  █ +0.9 - -0.1 · +0.0 · -0.0 · +0.0 · -0.0 ▒ +0.2 · +0.0 █ +1.0


In [ ]:
import numpy as np

C = np.corrcoef(df.values.T)
labels = df.columns.tolist()

pairs = []

for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        
        if labels[i] == "MedHouseVal" or labels[j] == "MedHouseVal":
            continue
        
        pairs.append((abs(C[i, j]), C[i, j], labels[i], labels[j]))

pairs.sort(reverse=True)

print("Top-3 pary cech:")
for abs_r, r, feature1, feature2 in pairs[:3]:
    print(f"{feature1} / {feature2}: r = {r:+.4f}")

Top-3 pary cech:
AveRooms / AveBedrms: r = +0.8751
AveRooms / AveOccup: r = +0.0479
AveRooms / Population: r = -0.0346


In [ ]:
zad 6

In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)


ages = df["Age"].dropna().values


n = len(ages)


mu0 = 30

x_mean = ages.mean()
x_std = ages.std(ddof=1)


t = (x_mean - mu0) / (x_std / np.sqrt(n))


p_value = 2 * (1 - normal_cdf(abs(t), 0, 1))

print(f"Liczba obserwacji: {n}")
print(f"Średni wiek: {x_mean:.4f}")
print(f"Odchylenie standardowe: {x_std:.4f}")
print(f"t = {t:.4f}")
print(f"p-value = {p_value:.6f}")

if p_value < 0.05:
    print("Odrzucamy H₀: średni wiek różni się od 30 lat.")
else:
    print("Nie odrzucamy H₀: brak podstaw, aby stwierdzić, że średni wiek różni się od 30 lat.")

Liczba obserwacji: 714
Średni wiek: 29.6991
Odchylenie standardowe: 14.5265
t = -0.5535
p-value = 0.579950
Nie odrzucamy H₀: brak podstaw, aby stwierdzić, że średni wiek różni się od 30 lat.


In [ ]:
zad 7

In [88]:
import pandas as pd
import numpy as np

def permutation_pvalue(a, b, n_perm=2000):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    observed = abs(a.mean() - b.mean())
    combined = np.concatenate([a, b])

    count = 0
    rng = np.random.default_rng(42)

    for _ in range(n_perm):
        shuffled = rng.permutation(combined)
        perm_a = shuffled[:len(a)]
        perm_b = shuffled[len(a):]

        difference = abs(perm_a.mean() - perm_b.mean())

        if difference >= observed:
            count += 1

    return (count + 1) / (n_perm + 1)


df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)

age_survived = df.loc[df["Survived"] == 1, "Age"].dropna().values
age_died = df.loc[df["Survived"] == 0, "Age"].dropna().values

n1, n2 = len(age_survived), len(age_died)

m1, m2 = age_survived.mean(), age_died.mean()

v1, v2 = age_survived.var(ddof=1), age_died.var(ddof=1)

se = np.sqrt(v1 / n1 + v2 / n2)

t = (m1 - m2) / se

p_value_perm = permutation_pvalue(
    age_survived,
    age_died,
    n_perm=2000
)

pooled_std = np.sqrt(
    ((n1 - 1) * v1 + (n2 - 1) * v2)
    / (n1 + n2 - 2)
)

cohens_d = (m1 - m2) / pooled_std

print(f"Przeżyli: n = {n1}, średnia = {m1:.2f}")
print(f"Nie przeżyli: n = {n2}, średnia = {m2:.2f}")
print(f"t-statystyka = {t:.4f}")
print(f"p-value = {p_value_perm:.6f}")
print(f"Cohen's d = {cohens_d:.4f}")

if p_value_perm < 0.05:
    print("WNIOSEK: Różnica wieku jest istotna statystycznie.")
else:
    print("WNIOSEK: Brak istotnej statystycznie różnicy wieku.")

Przeżyli: n = 290, średnia = 28.34
Nie przeżyli: n = 424, średnia = 30.63
t-statystyka = -2.0460
p-value = 0.040480
Cohen's d = -0.1575
WNIOSEK: Różnica wieku jest istotna statystycznie.


In [ ]:
zad 8

In [89]:
pd.crosstab(df['Pclass'], df['Survived'])

Survived,0,1
Pclass,,
1,80,136
2,97,87
3,372,119


In [92]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)


def chi2_monte_carlo_pvalue(observed, n_sim=5000, seed=42):
    rng = np.random.default_rng(seed)

    row_totals = observed.sum(axis=1)
    col_totals = observed.sum(axis=0)
    n = observed.sum()

    expected = np.outer(row_totals, col_totals) / n

    def chi2(table, exp):
        return ((table - exp) ** 2 / exp).sum()

    observed_chi2 = chi2(observed, expected)

    count_extreme = 0

    for _ in range(n_sim):
        probs = expected.flatten() / n

        table_sim = rng.multinomial(
            n, probs
        ).reshape(observed.shape)

        exp_sim = np.outer(
            table_sim.sum(axis=1),
            table_sim.sum(axis=0)
        ) / n

        exp_sim = np.maximum(exp_sim, 1e-9)

        if chi2(table_sim, exp_sim) >= observed_chi2:
            count_extreme += 1

    return count_extreme / n_sim


def ascii_stacked_bars(table):
    for index, row in table.iterrows():
        total = row.sum()

        survived = row.get(1, 0) / total * 100
        died = row.get(0, 0) / total * 100

        survived_bar = "█" * int(survived / 2)
        died_bar = "░" * int(died / 2)

        print(
            f"Pclass {index}: "
            f"{died_bar}{survived_bar} "
            f"zginęli={died:.1f}% "
            f"przeżyli={survived:.1f}%"
        )


observed = pd.crosstab(
    df["Pclass"],
    df["Survived"]
)


print("TABELA KONTYNGENCJI")
print(observed)


row_totals = observed.sum(axis=1).values
col_totals = observed.sum(axis=0).values
n = observed.values.sum()

expected = np.outer(
    row_totals,
    col_totals
) / n


chi2_stat = (
    (observed.values - expected) ** 2
    / expected
).sum()


r, c = observed.shape

dof = (r - 1) * (c - 1)


p_value = chi2_monte_carlo_pvalue(
    observed.values,
    n_sim=3000,
    seed=42
)


cramers_v = np.sqrt(
    chi2_stat
    / (
        n * min(r - 1, c - 1)
    )
)



print("TEST CHI-KWADRAT")


print("H₀: Przeżycie i Pclass są niezależne")
print("H₁: Przeżycie i Pclass są zależne")

print(f"Chi² = {chi2_stat:.4f}")
print(f"df = {dof}")
print(f"p-value = {p_value:.6f}")
print(f"Cramer's V = {cramers_v:.4f}")



print("STACKED BAR CHART")


ascii_stacked_bars(observed)





if p_value < 0.05:
    print("Odrzucamy H₀.")
    print("Przeżycie jest statystycznie zależne od klasy podróży Pclass.")
else:
    print("Nie odrzucamy H₀.")
    print("Brak podstaw do stwierdzenia zależności między Pclass a przeżyciem.")

TABELA KONTYNGENCJI
Survived    0    1
Pclass            
1          80  136
2          97   87
3         372  119
TEST CHI-KWADRAT
H₀: Przeżycie i Pclass są niezależne
H₁: Przeżycie i Pclass są zależne
Chi² = 102.8890
df = 2
p-value = 0.000000
Cramer's V = 0.3398
STACKED BAR CHART
Pclass 1: ░░░░░░░░░░░░░░░░░░███████████████████████████████ zginęli=37.0% przeżyli=63.0%
Pclass 2: ░░░░░░░░░░░░░░░░░░░░░░░░░░███████████████████████ zginęli=52.7% przeżyli=47.3%
Pclass 3: ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░████████████ zginęli=75.8% przeżyli=24.2%
Odrzucamy H₀.
Przeżycie jest statystycznie zależne od klasy podróży Pclass.


In [ ]:
zad 9


In [94]:
import numpy as np
import math

rng = np.random.default_rng(seed=42)

sim = rng.binomial(
    n=20,
    p=0.5,
    size=1000
)

counts = np.bincount(sim, minlength=21)

def binom_pmf(k, n, p):
    return math.comb(n, k) * p**k * (1 - p)**(n - k)

theoretical_pmf = np.array([
    binom_pmf(k, 20, 0.5)
    for k in range(21)
])

empirical_pmf = counts / len(sim)

def ascii_barplot_pmf(values, label, marker="█", scale=100):
    print(f"\n{label}")

    for k, value in enumerate(values):
        bar_length = int(value * scale)
        print(f"{k:2d} | {marker * bar_length} {value:.4f}")

ascii_barplot_pmf(
    empirical_pmf,
    "Symulacja",
    marker="█"
)

ascii_barplot_pmf(
    theoretical_pmf,
    "Teoretyczny PMF",
    marker="·"
)


print("PORÓWNANIE ŚREDNIEJ")


print(f"Średnia empiryczna:   {sim.mean():.4f}")
print(f"Średnia teoretyczna:  {20 * 0.5:.4f}")


Symulacja
 0 |  0.0000
 1 |  0.0000
 2 |  0.0000
 3 |  0.0010
 4 |  0.0060
 5 | █ 0.0130
 6 | ████ 0.0430
 7 | ███████ 0.0760
 8 | ████████████ 0.1240
 9 | ███████████████ 0.1510
10 | █████████████████ 0.1710
11 | ███████████████ 0.1560
12 | █████████████ 0.1330
13 | ███████ 0.0730
14 | ██ 0.0290
15 | █ 0.0180
16 |  0.0040
17 |  0.0020
18 |  0.0000
19 |  0.0000
20 |  0.0000

Teoretyczny PMF
 0 |  0.0000
 1 |  0.0000
 2 |  0.0002
 3 |  0.0011
 4 |  0.0046
 5 | · 0.0148
 6 | ··· 0.0370
 7 | ······· 0.0739
 8 | ············ 0.1201
 9 | ················ 0.1602
10 | ················· 0.1762
11 | ················ 0.1602
12 | ············ 0.1201
13 | ······· 0.0739
14 | ··· 0.0370
15 | · 0.0148
16 |  0.0046
17 |  0.0011
18 |  0.0002
19 |  0.0000
20 |  0.0000
PORÓWNANIE ŚREDNIEJ
Średnia empiryczna:   9.9780
Średnia teoretyczna:  10.0000


In [ ]:
zadanie 16

In [96]:
import pandas as pd
import numpy as np

def make_synthetic_wine(n_samples=1600, seed=42):
    """
    Kanoniczny syntetyczny dataset wina
    (wzorowany na UCI Wine Quality - red).
    11 cech fizykochemicznych + quality (3..8).
    """
    rng = np.random.default_rng(seed)

    alcohol = rng.normal(10.4, 1.1, size=n_samples).clip(8.4, 14.9)
    volatile_acidity = rng.gamma(
        shape=2.0,
        scale=0.25,
        size=n_samples
    ).clip(0.1, 1.6)

    sulphates = rng.gamma(
        shape=5.0,
        scale=0.13,
        size=n_samples
    ).clip(0.33, 2.0)

    citric_acid = rng.uniform(0.0, 0.75, size=n_samples)

    residual_sugar = rng.gamma(
        shape=2.0,
        scale=1.2,
        size=n_samples
    ).clip(0.9, 15.0)

    chlorides = rng.gamma(
        shape=3.0,
        scale=0.03,
        size=n_samples
    ).clip(0.012, 0.61)

    free_sulfur_dioxide = rng.gamma(
        shape=3.0,
        scale=5.0,
        size=n_samples
    ).clip(1, 72)

    total_sulfur_dioxide = (
        free_sulfur_dioxide
        * rng.uniform(2.0, 4.5, size=n_samples)
    )

    density = rng.normal(
        0.9967,
        0.0019,
        size=n_samples
    )

    pH = rng.normal(
        3.31,
        0.154,
        size=n_samples
    ).clip(2.74, 4.01)

    fixed_acidity = rng.gamma(
        shape=7.0,
        scale=1.2,
        size=n_samples
    ).clip(4.6, 15.9)

    quality_raw = (
        5.0
        + 0.32 * (alcohol - 10.4)
        - 1.15 * (volatile_acidity - 0.5)
        + 0.57 * (sulphates - 0.65)
        + 0.05 * (citric_acid - 0.3)
        + rng.normal(0, 0.5, size=n_samples)
    )

    quality = np.round(
        np.clip(quality_raw, 3, 8)
    ).astype(int)

    return pd.DataFrame({
        "fixed acidity": fixed_acidity,
        "volatile acidity": volatile_acidity,
        "citric acid": citric_acid,
        "residual sugar": residual_sugar,
        "chlorides": chlorides,
        "free sulfur dioxide": free_sulfur_dioxide,
        "total sulfur dioxide": total_sulfur_dioxide,
        "density": density,
        "pH": pH,
        "sulphates": sulphates,
        "alcohol": alcohol,
        "quality": quality
    })


wine = make_synthetic_wine(seed=42)

print("Syntetyczny dataset Wine Quality:")
print(wine.head())

print(f"\nShape: {wine.shape}")

print("\nOpis statystyczny:")
print(wine.describe().round(3))

Syntetyczny dataset Wine Quality:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0       5.337116          0.399187     0.152028        3.233766   0.273758   
1       5.555572          0.972553     0.016483        5.464401   0.024270   
2       7.879531          0.323766     0.735677        4.863200   0.176117   
3       5.640307          0.603101     0.194360        0.900000   0.088048   
4       6.335022          0.350284     0.444831        0.900000   0.059808   

   free sulfur dioxide  total sulfur dioxide   density        pH  sulphates  \
0            19.315469             66.358296  0.994562  3.086856   0.601784   
1            17.541303             74.924045  0.997984  3.148137   0.657098   
2             3.815322             12.458480  0.999426  3.362340   1.303008   
3             2.198062              8.108279  0.999149  3.420159   0.784769   
4             7.691284             28.947739  0.998058  3.186875   0.483775   

     alcohol  quality 

In [ ]:
import numpy as np

cols = [c for c in wine.columns if c != "quality"]
X = wine[cols].values

def corr_to_char(r):
    if r >= 0.75:
        return "█"
    elif r >= 0.5:
        return "▓"
    elif r >= 0.25:
        return "▒"
    elif r >= 0.05:
        return "░"
    elif r > -0.05:
        return "·"
    elif r > -0.25:
        return "-"
    elif r > -0.5:
        return "/"
    else:
        return "#"


def ascii_heatmap(M, labels):
    n = len(labels)
    short = [label[:7] for label in labels]

    print("\nASCII heatmap macierzy korelacji:")

    header = " " * 9 + "".join(
        f"{s:>7}" for s in short
    )
    print(header)

    for i in range(n):
        row = f"{short[i]:>8} "

        for j in range(n):
            row += f" {corr_to_char(M[i, j])} {M[i, j]:+.2f}"[:7]

        print(row)


C = np.corrcoef(X.T)

print("=" * 70)
print("MACIERZ KORELACJI PRZED USUWANIEM")
print("=" * 70)

ascii_heatmap(C, labels=cols)


removed = []

while True:
    C = np.corrcoef(X.T)

    n_features = C.shape[0]

    max_corr = 0
    best_i = None
    best_j = None

    for i in range(n_features):
        for j in range(i + 1, n_features):
            if abs(C[i, j]) > max_corr:
                max_corr = abs(C[i, j])
                best_i = i
                best_j = j

    if max_corr <= 0.7:
        break

    mean_corr_i = np.mean(
        np.abs(np.delete(C[i], i))
    )

    mean_corr_j = np.mean(
        np.abs(np.delete(C[j], j))
    )

    if mean_corr_i >= mean_corr_j:
        remove_idx = best_i
    else:
        remove_idx = best_j

    removed_feature = cols[remove_idx]

    removed.append(removed_feature)

    print(
        f"Usuwam: {removed_feature} "
        f"| max |r| = {max_corr:.4f}"
    )

    X = np.delete(X, remove_idx, axis=1)
    cols.pop(remove_idx)


C_final = np.corrcoef(X.T)

print("LISTA USUNIĘTYCH CECH")


for feature in removed:
    print(feature)

print(f"\nLiczba usuniętych cech: {len(removed)}")


print("MACIERZ KORELACJI PO USUNIĘCIU")


ascii_heatmap(C_final, labels=cols)

max_final_corr = np.max(
    np.abs(C_final - np.eye(C_final.shape[0]))
)


print("SPRAWDZENIE")


print(f"Pozostałe cechy: {len(cols)}")
print(f"Największe |r|: {max_final_corr:.4f}")
print(f"Wszystkie |r| <= 0.7: {max_final_corr <= 0.7}")

MACIERZ KORELACJI PRZED USUWANIEM

ASCII heatmap macierzy korelacji:
         fixed avolatilcitric residuachloridfree sutotal sdensity     pHsulphatalcohol
 fixed a  █ +1.0 · -0.0 · +0.0 · -0.0 · +0.0 · -0.0 · -0.0 · +0.0 · +0.0 · -0.0 · -0.0
 volatil  · -0.0 █ +1.0 · +0.0 · +0.0 · +0.0 · +0.0 · +0.0 · -0.0 · -0.0 · +0.0 · +0.0
 citric   · +0.0 · +0.0 █ +1.0 - -0.0 · +0.0 · -0.0 · -0.0 · -0.0 · -0.0 · +0.0 · +0.0
 residua  · -0.0 · +0.0 - -0.0 █ +1.0 · +0.0 · -0.0 · -0.0 ░ +0.0 · +0.0 · +0.0 · +0.0
 chlorid  · +0.0 · +0.0 · +0.0 · +0.0 █ +1.0 · +0.0 · -0.0 · -0.0 · -0.0 · +0.0 · +0.0
 free su  · -0.0 · +0.0 · -0.0 · -0.0 · +0.0 █ +1.0 █ +0.9 · -0.0 · -0.0 · -0.0 · +0.0
 total s  · -0.0 · +0.0 · -0.0 · -0.0 · -0.0 █ +0.9 █ +1.0 · -0.0 · -0.0 · -0.0 · +0.0
 density  · +0.0 · -0.0 · -0.0 ░ +0.0 · -0.0 · -0.0 · -0.0 █ +1.0 · -0.0 · -0.0 · +0.0
      pH  · +0.0 · -0.0 · -0.0 · +0.0 · -0.0 · -0.0 · -0.0 · -0.0 █ +1.0 · -0.0 · +0.0
 sulphat  · -0.0 · +0.0 · +0.0 · +0.0 · +0.0 · -0.0 · -0.0 · 